In [1]:
import pandas as pd
import matplotlib.pyplot as plt
##import warnings##python自带的功能警告
##warnings.filterwarnings("ignore")##无视警告

In [2]:
from sklearn.model_selection import train_test_split  #函数划分数据集
from sklearn.preprocessing import StandardScaler  #数据预处理 类 标准化器
from sklearn.linear_model import LogisticRegression   #模型 类 逻辑回归模型
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report  #评估指标 accuracy_score 算准确率 confusion_matrix 混淆矩阵  classification_report分类报告 

In [3]:
df=pd.read_csv("data/titanic_clean.csv")

In [4]:
#编码
df["Sex_num"]=df["Sex"].map({"male":0,"female":1})
df=pd.concat([df,pd.get_dummies(df["Embarked"],prefix="Emb")],axis=1)

In [5]:
#title
df["Title"]=df["Name"].str.extract(r",\s*([^.]+)\.")
rare=["Dr","Rev","Major","Col","the Countess","Capt",
      "Ms","Sir","Lady","Mme","Don","Jonkheer","Mlle"]
df["Title"]=df["Title"].replace(rare,"Rare")
df=pd.concat([df,pd.get_dummies(df["Title"],prefix="Title")],axis=1)

In [6]:
#FamilySize
df["FamilySize"]=df["SibSp"]+df["Parch"]+1
y=df["Survived"]

In [7]:
def evaluate_features(cols,name):
    X=df[cols]  #拿到数据
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)  #划分训练
    scaler=StandardScaler()  #造模型
    #fit训练
    X_train=scaler.fit_transform(X_train)
    X_test=scaler.transform(X_test)
    
    model=LogisticRegression(max_iter=1000)
    model.fit(X_train,y_train)
    #预测评估
    pred=model.predict(X_test) #predict算概率，按0.5划一刀判类别
    acc=accuracy_score(y_test,pred) #accuaracy_score 逐个对比，数相同的，除以总数

    print(f"{name:<28} 特征数{len(cols):>2}  准确率{acc:.4f}")
    return (acc, y_test, pred)
    

In [8]:
basic_feature=["Pclass", "Sex_num", "Age", "SibSp", "Parch",
            "Fare", "HasCabin", "Emb_C", "Emb_Q", "Emb_S"]
enhanced_feature=basic_feature+["FamilySize"]+[c for c in df.columns if c.startswith("Title_")]

evaluate_features(basic_feature,"只用原始字段")
evaluate_features(enhanced_feature,"加上 Title+FamilySize")

acc,y_test,pred=evaluate_features(enhanced_feature,"加强特征")

只用原始字段                       特征数10  准确率0.8090
加上 Title+FamilySize          特征数16  准确率0.8315
加强特征                         特征数16  准确率0.8315


相当于在 178 个测试样本里，多救对了 4 个人。
这就是特征工程的价值 —— 你不改模型、不改参数，只换特征，准确率就上去了。
而且现实工作中，调参带来的提升往往只有零点几个百分点，而好的特征能带来几个百分点。
这就是为什么特征工程比调参更值钱。

In [9]:
from sklearn.model_selection import cross_val_score
#对训练集做5折交叉验证，比单词划分稳得多
for name,cols in [("基础",basic_feature),("加强",enhanced_feature)]:
    scores=cross_val_score(
        LogisticRegression(max_iter=1000),
        StandardScaler().fit_transform(df[cols]),
        y,cv=5)
    print(f"{name:<8} 交叉验证平均{scores.mean():.4f}")

基础       交叉验证平均0.7987
加强       交叉验证平均0.8268


In [10]:
#混淆举证 看清楚错在哪
cm=confusion_matrix(y_test,pred)
print(cm)

[[91 18]
 [12 57]]


翻译成人话：
	        预测【遇难】	                预测【生还】
实际【遇难】	91 ✅ 猜对了	            18 ❌ 误报（死人被猜成活的）
实际【生还】	12 ❌ 漏报（活人被猜成死的）	57 ✅ 猜对了

即对角线上是正确的，其他都是预测错误
对角线越大越好  猜对的越多
反对角线越小越好 猜错的少
多分类也一样 对角线是对的


In [11]:
#四个评估指标
from sklearn.metrics import precision_score,recall_score,f1_score
print(f"accuracy (准确率):{accuracy_score(y_test,pred):.4f}")
print(f"precision (精确率):{precision_score(y_test,pred):.4f}")
print(f"recall (召回率):{recall_score(y_test,pred):.4f}")
print(f"f1 (f1分数):{f1_score(y_test,pred):.4f}")

accuracy (准确率):0.8315
precision (精确率):0.7600
recall (召回率):0.8261
f1 (f1分数):0.7917


用上面的数字手算一遍，你就懂了：
指标	        公式	算出来	人话
accuracy	(91+57)/178	0.8315	全部样本里，猜对多少
precision	57/(57+18)	0.7600	我预测"生还"的人里，多少真的生还了
recall	    57/(57+12)	0.8261	真正生还的人里，我找出来多少
f1	      两者的调和平均	0.7917	上面两个的平衡分

到底该看哪个？看你的目标
场景	    更该看	    为什么
癌症筛查	    recall	    宁可误报（多查一次），绝不能漏诊
垃圾邮件过滤	precision	宁可漏掉几封垃圾邮件，也不能把重要邮件扔进垃圾箱
泰坦尼克预测	看你想干什么	想"尽量多救人"看 recall；想"判断精准"看 precision


一句话记住：
- precision = "我说是的里面，有多少真的是"（别误报）
- recall = "真的是里面，我找出来多少"（别漏报）

In [12]:
#一行代码看全
print(classification_report(y_test,pred,target_names=["遇难","生还"]))  #classification_report 是sklearn 的函数：生成分类报告

              precision    recall  f1-score   support

          遇难       0.88      0.83      0.86       109
          生还       0.76      0.83      0.79        69

    accuracy                           0.83       178
   macro avg       0.82      0.83      0.83       178
weighted avg       0.84      0.83      0.83       178



In [13]:
# ===== ① 在外面准备数据（用加强特征）=====
X = df[enhanced_feature]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== ② 标准化（★ 关键一步）=====
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)     # 训练集：fit + transform
X_test_scaled  = scaler.transform(X_test)          # 测试集：只 transform

# ===== ③ 逻辑回归 =====
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)
print("逻辑回归:", round(accuracy_score(y_test, pred_lr), 4))

# ===== ④ KNN（用【同一份】标准化数据）=====
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
pred_knn = knn.predict(X_test_scaled)
print("KNN:     ", round(accuracy_score(y_test, pred_knn), 4))

逻辑回归: 0.8315


NameError: name 'KNeighborsClassifier' is not defined

1	为什么"加强特征"比"基础特征"表现好？	想想 Title 和 FamilySize 提供了什么信息
2	混淆矩阵里，18 和 12 各代表什么？	一个是"误报"，一个是"漏报"
3	如果这是癌症筛查，你会更关注 precision 还是 recall？为什么？	想想"漏诊"的代价

1	加强特征多了 Title 和 FamilySize 两个特征。Title 补上了 Sex 分不清的年龄信息（Master 和 Mr 都是 male，但生还率差 4 倍）；FamilySize 把 SibSp+Parch 综合成"家庭规模"。给模型提供了更多能用的信息
2	18 = 误报（FP）：实际遇难被猜成生还；12 = 漏报（FN）：实际生还被猜成遇难
3	看 recall，因为癌症筛查最怕漏诊（有病的没查出来）。注意是"漏诊"不是"误诊" —— 误诊（误报）对应的是 precision